# PhytoSense AR – train the disease model on real field photos

The original model was trained only on **PlantVillage** (lab photos, one leaf on a plain background),
so it fails on real photos. This notebook fine-tunes it on **PlantVillage + PlantDoc**
(2,300+ real field photos with cluttered backgrounds) with strong augmentation, and measures the
improvement on the **PlantDoc test set** (real-world photos never seen in training).

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Takes about 30–45 minutes. At the end it downloads `phytosense_model.zip`
(`model.onnx` + `labels.json` + `metrics.json`). Unzip it into `PhytoSense-AR/backend/model/`
to replace the old model. No app code changes are needed, because the labels and input format are the same.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install onnx onnxruntime
import torch
assert torch.cuda.is_available(), "Select a GPU runtime: Runtime -> Change runtime type -> T4 GPU"

## 1. Download the datasets

In [ ]:
%cd /content
# PlantDoc: real field photos (train/ and test/ folders, one folder per class)
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git
# PlantVillage: only the colour images (sparse checkout keeps the download ~1 GB)
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git
!cd PlantVillage-Dataset && git sparse-checkout set raw/color
!ls PlantDoc-Dataset/train | wc -l; ls PlantVillage-Dataset/raw/color | wc -l

## 2. Settings and label mapping (PlantDoc class → app label)

In [ ]:
import json, random, re
from pathlib import Path
from transformers import AutoModelForImageClassification

SEED = 42
BASE_MODEL = "kimcomehome/plantvillage-vit-leaf-disease"   # the model the app uses today
PV_PER_CLASS = 250        # PlantVillage images per class used for training (keeps old knowledge)
PV_TEST_PER_CLASS = 20    # held-out PlantVillage images to check nothing was forgotten
PD_REPEAT = 4             # PlantDoc is small, so show each field photo 4x per epoch (different augmentations)
EPOCHS = 8
BATCH = 32
LR = 3e-5

random.seed(SEED); torch.manual_seed(SEED)
PD = Path("/content/PlantDoc-Dataset"); PV = Path("/content/PlantVillage-Dataset/raw/color")

base = AutoModelForImageClassification.from_pretrained(BASE_MODEL)
LABELS = [base.config.id2label[i] for i in range(len(base.config.id2label))]
L2I = {l: i for i, l in enumerate(LABELS)}
print(len(LABELS), "labels")

PLANTDOC_TO_LABEL = {
    "apple scab leaf": "Apple___Apple_scab",
    "apple leaf": "Apple___healthy",
    "apple rust leaf": "Apple___Cedar_apple_rust",
    "bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
    "bell_pepper leaf": "Pepper,_bell___healthy",
    "blueberry leaf": "Blueberry___healthy",
    "cherry leaf": "Cherry_(including_sour)___healthy",
    "corn gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "corn rust leaf": "Corn_(maize)___Common_rust_",
    "peach leaf": "Peach___healthy",
    "potato leaf": "Potato___healthy",
    "potato leaf early blight": "Potato___Early_blight",
    "potato leaf late blight": "Potato___Late_blight",
    "raspberry leaf": "Raspberry___healthy",
    "soyabean leaf": "Soybean___healthy",
    "squash powdery mildew leaf": "Squash___Powdery_mildew",
    "strawberry leaf": "Strawberry___healthy",
    "tomato early blight leaf": "Tomato___Early_blight",
    "tomato septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "tomato leaf late blight": "Tomato___Late_blight",
    "tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "tomato leaf": "Tomato___healthy",
    "tomato mold leaf": "Tomato___Leaf_Mold",
    "tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "grape leaf black rot": "Grape___Black_rot",
    "grape leaf": "Grape___healthy",
}
assert all(v in L2I for v in PLANTDOC_TO_LABEL.values())
norm = lambda s: re.sub(r"\s+", " ", s.strip().lower())
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def plantdoc_items(split):
    items = []
    for d in sorted((PD / split).iterdir()):
        if not d.is_dir():
            continue
        label = PLANTDOC_TO_LABEL.get(norm(d.name))
        assert label, f"Unmapped PlantDoc class: {d.name!r}"
        items += [(p, L2I[label]) for p in d.iterdir() if p.suffix.lower() in IMG_EXT]
    return items

pd_train_all, pd_test = plantdoc_items("train"), plantdoc_items("test")
random.shuffle(pd_train_all)
n_val = len(pd_train_all) // 10
pd_val, pd_train = pd_train_all[:n_val], pd_train_all[n_val:]

pv_train, pv_test = [], []
for label in LABELS:
    files = sorted(p for p in (PV / label).iterdir() if p.suffix.lower() in IMG_EXT); random.shuffle(files)
    pv_test += [(p, L2I[label]) for p in files[:PV_TEST_PER_CLASS]]
    pv_train += [(p, L2I[label]) for p in files[PV_TEST_PER_CLASS:PV_TEST_PER_CLASS + PV_PER_CLASS]]

print(f"PlantDoc train {len(pd_train)}, val {len(pd_val)}, test {len(pd_test)} | "
      f"PlantVillage train {len(pv_train)}, test {len(pv_test)}")

## 3. Shrink images once (PlantDoc photos are large; this makes training ~5x faster)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
CACHE = Path("/content/cache"); CACHE.mkdir(exist_ok=True)

def cache_one(item):
    src, y = item
    dst = CACHE / (str(abs(hash(str(src)))) + ".jpg")
    if not dst.exists():
        try:
            im = Image.open(src).convert("RGB")
            im.thumbnail((384, 384), Image.BILINEAR)
            im.save(dst, quality=92)
        except Exception as e:
            print("skip", src.name, e); return None
    return (dst, y)

def cache(items):
    with ThreadPoolExecutor(8) as ex:
        return [r for r in ex.map(cache_one, items) if r]

pd_train, pd_val, pd_test, pv_train, pv_test = map(cache, (pd_train, pd_val, pd_test, pv_train, pv_test))
print("cached", len(list(CACHE.iterdir())), "images")

## 4. Data pipeline with field-style augmentation

In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

def app_preprocess(im):
    # Exactly what backend/app/classifier.py does: 224x224 bilinear, scale to [-1, 1], CHW.
    x = np.asarray(im.resize((224, 224), Image.BILINEAR), dtype=np.float32) / 127.5 - 1.0
    return torch.from_numpy(x.transpose(2, 0, 1))

train_aug = T.Compose([
    T.RandomResizedCrop(224, scale=(0.25, 1.0), ratio=(0.7, 1.4)),   # partial leaves, zoom levels
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(25),
    T.ColorJitter(0.4, 0.4, 0.3, 0.03),                               # sun, shade, phone cameras
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),      # out-of-focus shots
])

class Images(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        im = Image.open(p).convert("RGB")
        if self.train:
            im = train_aug(im)
        return app_preprocess(im), y

train_items = pv_train + pd_train * PD_REPEAT
loader = lambda items, train: DataLoader(Images(items, train), batch_size=BATCH if train else 64,
                                         shuffle=train, num_workers=2, pin_memory=True, drop_last=train)
train_dl = loader(train_items, True)
print(len(train_items), "training samples per epoch")

## 5. Baseline: how good is the current model on real photos?

In [ ]:
from collections import Counter

@torch.no_grad()
def evaluate(model, items):
    model.eval(); correct = 0; preds = []
    for x, y in loader(items, False):
        with torch.autocast("cuda", dtype=torch.float16):
            p = model(pixel_values=x.cuda()).logits.argmax(1).cpu()
        correct += (p == y).sum().item(); preds += p.tolist()
    return correct / len(items), preds

base = base.cuda()
base_pd, _ = evaluate(base, pd_test)
base_pv, _ = evaluate(base, pv_test)
print(f"BASELINE  PlantDoc test (real photos): {base_pd:.1%} | PlantVillage test (lab photos): {base_pv:.1%}")

## 6. Fine-tune

In [ ]:
import copy, math, time
from transformers import get_cosine_schedule_with_warmup

model = base
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
steps = EPOCHS * len(train_dl)
sched = get_cosine_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.amp.GradScaler("cuda")
loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

best_acc, best_state, history = -1, None, []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); total = 0
    for x, y in train_dl:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(pixel_values=x).logits, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    val_acc, _ = evaluate(model, pd_val)
    history.append({"epoch": epoch, "loss": total / len(train_dl), "plantdoc_val_acc": val_acc})
    print(f"epoch {epoch}: loss {total/len(train_dl):.3f}  PlantDoc val {val_acc:.1%}  ({time.time()-t0:.0f}s)")
    if val_acc > best_acc:
        best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)

## 7. Final results (for the project report)

In [ ]:
new_pd, pd_preds = evaluate(model, pd_test)
new_pv, _ = evaluate(model, pv_test)
print(f"                     before   after")
print(f"PlantDoc (real)      {base_pd:6.1%}  {new_pd:6.1%}")
print(f"PlantVillage (lab)   {base_pv:6.1%}  {new_pv:6.1%}")

# Per-class accuracy on real photos
per = {}
for (_, y), p in zip(pd_test, pd_preds):
    c = per.setdefault(LABELS[y], [0, 0]); c[0] += p == y; c[1] += 1
for k, (ok, n) in sorted(per.items()):
    print(f"  {k:55s} {ok:3d}/{n:<3d} {ok/n:6.1%}")

## 8. Export int8 ONNX (same format the app loads) and check it

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

OUT = Path("/content/export"); OUT.mkdir(exist_ok=True)
cpu = model.float().cpu().eval()
torch.onnx.export(cpu, (torch.zeros(1, 3, 224, 224),), str(OUT / "fp32.onnx"), input_names=["pixel_values"],
                  output_names=["logits"], opset_version=17, dynamo=False)
quantize_dynamic(str(OUT / "fp32.onnx"), str(OUT / "model.onnx"), weight_type=QuantType.QUInt8)
for f in OUT.glob("fp32.onnx*"):
    f.unlink()

sess = ort.InferenceSession(str(OUT / "model.onnx"), providers=["CPUExecutionProvider"])
def onnx_acc(items):
    ok = 0
    for p, y in items:
        x = app_preprocess(Image.open(p).convert("RGB"))[None].numpy()
        ok += int(sess.run(None, {"pixel_values": x})[0].argmax()) == y
    return ok / len(items)
int8_pd, int8_pv = onnx_acc(pd_test), onnx_acc(pv_test)
print(f"int8 ONNX: PlantDoc {int8_pd:.1%} | PlantVillage {int8_pv:.1%} | "
      f"{(OUT/'model.onnx').stat().st_size/1e6:.0f} MB")

(OUT / "labels.json").write_text(json.dumps(LABELS, indent=0), encoding="utf-8")
(OUT / "metrics.json").write_text(json.dumps({
    "base_model": BASE_MODEL, "train_data": "PlantVillage subset + PlantDoc train (field photos)",
    "plantdoc_test": {"before": base_pd, "after": new_pd, "after_int8_onnx": int8_pd, "n": len(pd_test)},
    "plantvillage_test": {"before": base_pv, "after": new_pv, "after_int8_onnx": int8_pv, "n": len(pv_test)},
    "per_class_plantdoc": {k: ok / n for k, (ok, n) in per.items()},
    "history": history,
}, indent=2), encoding="utf-8")

In [ ]:
!cd /content/export && zip -q /content/phytosense_model.zip model.onnx labels.json metrics.json
from google.colab import files
files.download("/content/phytosense_model.zip")